In [1]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torchvision.transforms import transforms
from torchvision.datasets import ImageFolder
from PIL import Image


In [2]:
!pip install kagglehub

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tawsifurrahman/tuberculosis-tb-chest-xray-dataset")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Lenovo\.cache\kagglehub\datasets\tawsifurrahman\tuberculosis-tb-chest-xray-dataset\versions\3


In [4]:
data_transforms = transforms.Compose([
    transforms.Resize([512,512]),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.43,0.49,0.41],std = [0.23,0.26,0.29])
]
)

In [5]:
import os

In [6]:
class SimpleImageDataset(Dataset):
    def __init__(self,data_dir,transform = None):
        self.data = []
        self.transform = transform 
        
        self.classes = sorted([
    d for d in os.listdir(data_dir)
    if os.path.isdir(os.path.join(data_dir, d))
])

        self.classes_to_idx = {c:i for i, c in enumerate(self.classes)}
        for cls in self.classes:
            cls_path = os.path.join(data_dir,cls)
            if not os.path.isdir(cls_path):
                continue
            
            for img in os.listdir(cls_path):
                if img.lower().endswith(('.jpg','.jpeg','.png','.webp')):
                    self.data.append((
                        os.path.join(cls_path,img),self.classes_to_idx[cls]
                    ))
        print("classes:", self.classes)
        print("Total images:", len(self.data))
                    
    def __len__(self):
        return len(self.data)
    def __getitem__(self,idx):
        image_path,label = self.data[idx]
        image = Image.open(image_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image,label

In [7]:
dataset = SimpleImageDataset(
    r'C:\Users\Lenovo\OneDrive\Desktop\Data Structures and Algorithms\Deep Learning\Pytorch\TB Chest Dataset\TB_Chest_Radiography_Database',
    transform=data_transforms
)

print("Dataset length:", len(dataset))


classes: ['Normal', 'Tuberculosis']
Total images: 4200
Dataset length: 4200


In [8]:
len(dataset)

4200

In [9]:
print(dataset.classes)

['Normal', 'Tuberculosis']


In [10]:
from torch.utils.data import random_split
train_dataset = int(0.8*len(dataset))
test_dataset = len(dataset) - train_dataset

train_dataset,test_dataset = random_split(dataset,[train_dataset,test_dataset])

In [11]:
train_loader = DataLoader(train_dataset, batch_size = 64, shuffle=True)
test_loader = DataLoader(test_dataset , batch_size = 64)

In [12]:
class CNNModel(nn.Module):
    def __init__(self,in_channels):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels,out_channels=16,kernel_size=3,stride = 1,padding='same'),
            nn.ReLU(),
            
            nn.MaxPool2d(kernel_size=2,stride=2),
            nn.Conv2d(16,32,kernel_size=3,stride=1,padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2)
            
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32*128*128,512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512,216),
            nn.ReLU(),
            nn.Dropout(0.6),
            nn.Linear(216,128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128,2)
            
        )
    def forward(self,x):
        x = self.features(x)
        y = self.classifier(x)
        return y

In [13]:
device = torch.device('cuda' if torch.cuda.is_available else 'cpu' )
model = CNNModel(3).to(device)

In [14]:
import torch.nn as nn
import torch.optim as optim

In [15]:
epochs = 10
learning_rate = 0.001
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)


In [16]:
model.train()
for epoch in range(epochs):
    total_loss = 0
    for image,label in train_loader:
        image,label = image.to(device),label.to(device)
        
        optimizer.zero_grad()
        pred = model(image)
        loss = criterion(pred,label)
        loss.backward()
        optimizer.step()
        total_loss +=  loss.item()
    avg_loss = total_loss/len(train_loader)
    print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}')
    
        
        

Epoch [1/10], Loss: 0.5550
Epoch [2/10], Loss: 0.1146
Epoch [3/10], Loss: 0.0699
Epoch [4/10], Loss: 0.0355
Epoch [5/10], Loss: 0.0178
Epoch [6/10], Loss: 0.0087
Epoch [7/10], Loss: 0.0205
Epoch [8/10], Loss: 0.0077
Epoch [9/10], Loss: 0.0441
Epoch [10/10], Loss: 0.0132


In [17]:
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for image, label in test_loader:
        image = image.to(device)
        label = label.to(device)

        outputs = model(image)
        _, predicted = torch.max(outputs, dim=1)

        total += label.size(0)
        correct += (predicted == label).sum().item()

accuracy = correct / total
print(f"Test Accuracy: {accuracy * 100:.2f}%")


Test Accuracy: 99.52%


In [20]:
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        outputs = model(image)
        _, predicted = torch.max(outputs, dim=1)

        total += label.size(0)
        correct += (predicted == label).sum().item()

accuracy = correct / total
print(f"Test Accuracy: {accuracy * 100:.2f}%")


Test Accuracy: 100.00%


In [26]:
model.eval()

image_path = r'C:\Users\Lenovo\OneDrive\Desktop\Data Structures and Algorithms\Deep Learning\Pytorch\others (103).jpg'

image = Image.open(image_path).convert('RGB')
image = data_transforms(image).unsqueeze(0).to(device)

class_names = dataset.classes   # IMPORTANT

with torch.no_grad():
    outputs = model(image)
    probs = torch.softmax(outputs, dim=1)
    predicted = torch.argmax(probs, dim=1)

predicted_class = class_names[predicted.item()]
confidence = probs[0][predicted.item()].item()

print("Predicted class:", predicted_class)
print(f"Confidence: {confidence*100:.2f}%")


Predicted class: Normal
Confidence: 99.98%
